# **Wstępna Eksploracyjna Analiza Danych (EDA): Dane Surowe**

---

### **Cel Analizy**

Niniejszy notatnik przedstawia wstępną eksploracyjną analizę danych (EDA) surowych, nieprzetworzonych danych, zanim zostaną one poddane głównemu potokowi przetwarzania. Celem jest zrozumienie dwóch kluczowych aspektów materiału źródłowego:

1.  **Dystrybucja Długości Tekstu:** Analiza długości raportów (w znakach), uzasadniająca potrzebę użycia modelu o długim oknie kontekstowym.
2.  **Dystrybucja Oryginalnych Ocen Eksperckich:** Analiza natury ocen eksperckich dla wszystkich 9 kryteriów ESG, co stanowi uzasadnienie dla naszego hybrydowego podejścia do modelowania.

### **1. Konfiguracja i Wczytanie Danych**

In [1]:
import pandas as pd
import plotly.express as px
import json
from pathlib import Path

# --- Konfiguracja ---
# Ta analiza wykorzystuje surowe oceny ekspertów i oczyszczone teksty.
RAW_SCORES_PATH = '../data/expert_scores.csv'
CLEAN_JSONL_PATH = '../data/processed/reports_texts_clean.jsonl'

In [2]:
# --- Wczytanie i Połączenie Danych ---
# Wczytanie ocen ekspertów
try:
    scores_df = pd.read_csv(RAW_SCORES_PATH, encoding='utf-8', sep=";")
    scores_df.columns = scores_df.columns.str.strip()
    scores_df.set_index('Raport', inplace=True)
    # Konwersja przecinków dziesiętnych na kropki
    for col in scores_df.columns:
        if scores_df[col].dtype == 'object':
            scores_df[col] = scores_df[col].str.replace(',', '.').astype(float)
    print(f"✅ Wczytano {len(scores_df)} rekordów z pliku CSV z ocenami ekspertów.")
except FileNotFoundError:
    print(f"❌ Nie znaleziono pliku z ocenami w: {RAW_SCORES_PATH}")
    scores_df = pd.DataFrame()

# Wczytanie długości tekstów
reports_list = []
try:
    with open(CLEAN_JSONL_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            reports_list.append(json.loads(line))
    text_lengths_df = pd.DataFrame({
        'Raport': report.get('nazwa_raportu', '').strip(),
        'text_length': len(report.get('text', ''))
    } for report in reports_list).set_index('Raport')
    print(f"✅ Wczytano {len(text_lengths_df)} rekordów z oczyszczonego pliku JSONL.")
    
    # Połączenie ramek danych
    if not scores_df.empty:
        raw_data_df = scores_df.join(text_lengths_df, how='inner')
        print(f"✅ Połączono dane. Finalna ramka danych zawiera {len(raw_data_df)} rekordów.")
except FileNotFoundError:
    print(f"❌ Nie znaleziono oczyszczonego pliku JSONL w: {CLEAN_JSONL_PATH}")
    raw_data_df = pd.DataFrame()

✅ Wczytano 393 rekordów z pliku CSV z ocenami ekspertów.
✅ Wczytano 393 rekordów z oczyszczonego pliku JSONL.
✅ Połączono dane. Finalna ramka danych zawiera 393 rekordów.


<br>

---

### **2. Analiza Długości Surowych Tekstów**

Poniższy histogram przedstawia rozkład liczby znaków w raportach korporacyjnych po wstępnym oczyszczeniu, ale przed tokenizacją.

In [ ]:
if not raw_data_df.empty:
    fig = px.histogram(
        raw_data_df,
        x='text_length',
        title='<b>Dystrybucja Długości Surowych Raportów (w znakach)</b>',
        labels={'text_length': 'Długość raportu (liczba znaków)'},
        nbins=50
    )
    fig.update_layout(
        yaxis_title='Liczba raportów',
        title_x=0.5,
        bargap=0.1
    )
    fig.show()

#### **Wniosek:**
Dystrybucja jest prawostronnie skośna, z większością raportów o długości od 0 do 400 000 znaków. Długi ogon rozkładu potwierdza obecność kilku bardzo obszernych raportów, co podkreśla konieczność zastosowania architektury modelu zdolnej do przetwarzania długich dokumentów, takiej jak Longformer.

<br>

---

### **3. Analiza Oryginalnych Ocen Eksperckich**

Ta sekcja analizuje strukturę oryginalnych ocen eksperckich. Ujawnia ona kluczowe rozróżnienie między dwoma typami kryteriów w naszym zbiorze danych, co bezpośrednio uzasadnia nasze hybrydowe podejście do modelowania.

In [ ]:
if not raw_data_df.empty:
    melted_df = raw_data_df.melt(
        id_vars=['text_length'], 
        value_vars=[str(i) for i in range(1, 10)], 
        var_name='Kryterium', 
        value_name='Ocena'
    )
    
    # --- WYKRES 1: KRYTERIA BINARNE ---
    binary_criteria_df = melted_df[melted_df['Kryterium'] != '3']
    
    print("--- Dystrybucja dla kryteriów binarnych (Oceny 0 lub 1) ---")
    fig1 = px.histogram(
        binary_criteria_df,
        x='Ocena',
        facet_col='Kryterium',
        facet_col_wrap=4,
        height=500,
        title='<b>Dystrybucja Oryginalnych Ocen dla Kryteriów Binarnych</b>',
        labels={'count': 'Liczba raportów', 'Ocena': 'Ocena ekspercka (0 lub 1)'}
    )
    fig1.update_layout(title_x=0.5, yaxis_title='')
    fig1.for_each_annotation(lambda a: a.update(text=f"<b>Kryterium {a.text.split('=')[-1]}</b>"))
    fig1.update_xaxes(type='category')
    fig1.show()

    # --- WYKRES 2: KRYTERIUM WIELOPOZIOMOWE ---
    multi_level_criterion_df = melted_df[melted_df['Kryterium'] == '3']
    
    print("\n--- Dystrybucja dla kryterium wielopoziomowego (Kryterium 3) ---")
    fig2 = px.histogram(
        multi_level_criterion_df,
        x='Ocena',
        title='<b>Dystrybucja Oryginalnych Ocen dla Wielopoziomowego Kryterium 3</b>',
        labels={'count': 'Liczba raportów', 'Ocena': 'Ocena ekspercka (0 do 1)'},
        text_auto=True
    )
    fig2.update_layout(title_x=0.5, bargap=0.2)
    fig2.update_xaxes(type='category')
    fig2.show()

--- Dystrybucja dla kryteriów binarnych (Oceny 0 lub 1) ---



--- Dystrybucja dla kryterium wielopoziomowego (Kryterium 3) ---


#### **Kluczowe Wnioski i Uzasadnienie Modelu Hybrydowego:**

Powyższe wizualizacje wyraźnie ukazują dwie odmienne metodologie oceny w naszych danych surowych:

1.  **Kryteria Binarne (0 lub 1):** Jak widać na pierwszym panelu wykresów, osiem kryteriów (1, 2, 4, 5, 6, 7, 8, 9) ma charakter fundamentalnie binarny. Oceny eksperckie to wyłącznie 0 lub 1, co reprezentuje jednoznaczną odpowiedź „nie” lub „tak”. Tego typu zadania są idealnie dopasowane do standardowego modelu klasyfikacji binarnej/wieloetykietowej.

2.  **Kryterium Wielopoziomowe, Hierarchiczne (Kryterium 3):** Drugi wykres podkreśla unikalny charakter Kryterium 3. Posiada ono granularny, wielopoziomowy system ocen (0, 0.25, 0.5, 0.75, 1.0), odzwierciedlający hierarchiczną ocenę dojrzałości ujawnień. Ten typ ustrukturyzowanej, opartej na regułach oceny nie jest optymalnym zadaniem probabilistycznego modelu głębokiego uczenia, którego celem jest klasyfikacja binarna.

**To fundamentalne rozróżnienie jest głównym uzasadnieniem dla architektury naszego systemu hybrydowego:**
-   **Model Uczenia Maszynowego** (`Longformer`) jest przeznaczony do rozwiązywania sześciu najbardziej złożonych, binarnych problemów klasyfikacyjnych.
-   Deterministyczny **System Oparty na Regułach** jest używany do obsługi trzech kryteriów (w tym wielopoziomowego Kryterium 3), które bazują na precyzyjnych, weryfikowalnych warunkach, zapewniając 100% dokładność i przejrzystość dla tych konkretnych zadań.